# YouTube Tool-Calling Agent with LangChain

This project demonstrates how to build an AI agent that interacts with YouTube programmatically using **tool calling** — a technique that lets a Large Language Model (LLM) decide when and how to use external functions to answer a query.

The agent is built with **LangChain** and powered by **OpenAI's GPT-4o-mini**. It can:
- Extract video IDs from any YouTube URL format
- Fetch video transcripts
- Search YouTube by keyword
- Extract rich metadata (views, likes, duration, chapters)
- Retrieve video thumbnails
- Generate summaries — all driven by the LLM's own reasoning

Three levels of agent complexity are covered: **manual tool calling**, **automated chains**, and a fully **recursive agent loop**.

## Table of Contents

1. [Setup](#setup)
2. [Building the Tools](#tools)
3. [Binding Tools to the LLM](#binding)
4. [Manual Tool Calling](#manual)
5. [Automated Summarization Chain](#chain)
6. [Recursive Agent Loop](#recursive)

---
## 1. Setup <a id='setup'></a>

In [ ]:
%%capture
%pip install pytube
%pip install youtube-transcript-api==1.1.0
%pip install langchain-community==0.3.16
%pip install langchain==0.3.23
%pip install langchain-openai==0.3.14
%pip install yt-dlp

In [ ]:
import os
import re
import json
import warnings
import logging

from typing import List, Dict
from pytube import YouTube
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from IPython.display import display, JSON
import yt_dlp

# Suppress noisy output
warnings.filterwarnings("ignore")
logging.getLogger('pytube').setLevel(logging.ERROR)
yt_dpl_logger = logging.getLogger('yt_dlp')
yt_dpl_logger.setLevel(logging.ERROR)

### Initialize the LLM

`init_chat_model` is a LangChain factory function that creates a provider-agnostic LLM object. Swapping `model_provider` to `"anthropic"` or `"google"` requires no other code changes.

Set your OpenAI API key as an environment variable before running:

In [ ]:
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

---
## 2. Building the Tools <a id='tools'></a>

A **tool** in LangChain is a regular Python function decorated with `@tool`. The decorator does three things:
- Registers the function with LangChain's tool system
- Generates a JSON schema from the type annotations (for input validation)
- Exposes the docstring to the LLM — this is how the model knows when and how to use the tool

```python
@tool
def tool_name(param: type) -> return_type:
    """The LLM reads this to decide when to call the tool."""
    ...
```

### Tool 1 — Extract Video ID

YouTube URLs come in three formats. A regex pattern handles all of them and extracts the 11-character video ID, which is required by most YouTube APIs.

In [ ]:
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts the 11-character YouTube video ID from a URL.

    Args:
        url (str): A YouTube URL (standard, shortened, or embedded format).

    Returns:
        str: The extracted video ID, or an error message if parsing fails.
    """
    pattern = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else "Error: Invalid YouTube URL"

In [ ]:
# Verify the tool is correctly registered
print("Name:", extract_video_id.name)
print("Description:", extract_video_id.description)

# Test execution
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")

In [ ]:
tools = [extract_video_id]

### Tool 2 — Fetch Transcript

Uses the `YouTubeTranscriptApi` library to pull the spoken transcript of a video. Accepts an optional language code (defaults to English).

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

@tool
def fetch_transcript(video_id: str, language: str = "en") -> str:
    """
    Fetches the transcript of a YouTube video.

    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (default: "en").

    Returns:
        str: The full transcript as a single string, or an error message.
    """
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id, languages=[language])
        return " ".join([snippet.text for snippet in transcript.snippets])
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
fetch_transcript.run("hfIUstzHs9A")

In [ ]:
tools.append(fetch_transcript)

### Tool 3 — Search YouTube

Uses PyTube's `Search` class to query YouTube and return a list of matching videos with their title, ID, and URL.

In [ ]:
from pytube import Search
from langchain.tools import tool

@tool
def search_youtube(query: str) -> List[Dict[str, str]]:
    """
    Search YouTube for videos matching the query.

    Args:
        query (str): The search term.

    Returns:
        List of dicts with 'title', 'video_id', and 'url' for each result.
    """
    try:
        s = Search(query)
        return [
            {
                "title": yt.title,
                "video_id": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }
            for yt in s.results
        ]
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
search_out = search_youtube.run("Generative AI")
display(JSON(search_out))

In [ ]:
tools.append(search_youtube)

### Tool 4 — Get Full Metadata

Uses `yt-dlp` to extract comprehensive metadata from a video URL without downloading the file. The `download=False` flag is key — it fetches only the info dictionary.

In [ ]:
@tool
def get_full_metadata(url: str) -> dict:
    """Extract metadata from a YouTube URL: title, views, duration, channel, likes, comments, and chapters."""
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            'title': info.get('title'),
            'views': info.get('view_count'),
            'duration': info.get('duration'),
            'channel': info.get('uploader'),
            'likes': info.get('like_count'),
            'comments': info.get('comment_count'),
            'chapters': info.get('chapters', [])
        }

In [ ]:
meta_data = get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")
display(JSON(meta_data))

In [ ]:
tools.append(get_full_metadata)

### Tool 5 — Get Thumbnails

Returns all available thumbnail URLs for a given video.

In [ ]:
@tool
def get_thumbnails(url: str) -> List[Dict]:
    """Retrieve all available thumbnail URLs for a YouTube video."""
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
        info = ydl.extract_info(url, download=False)
        return info.get('thumbnails', [])

In [ ]:
tools.append(get_thumbnails)
print(f"{len(tools)} tools registered:", [t.name for t in tools])

---
## 3. Binding Tools to the LLM <a id='binding'></a>

Calling `llm.bind_tools(tools)` injects the tool schemas into the LLM's context. From this point, when the model is invoked, it can either reply directly or emit a **tool call** — a structured request asking your code to run a specific function with specific arguments.

A `tool_mapping` dictionary is also created to look up the correct function by name at execution time.

In [ ]:
llm_with_tools = llm.bind_tools(tools)

# Map tool names to their callable objects
tool_mapping = {t.name: t for t in tools}
print(tool_mapping)

---
## 4. Manual Tool Calling <a id='manual'></a>

Before automating, it's worth walking through the tool-calling loop by hand. This makes the mechanics fully transparent:

1. Send a query → LLM decides to call a tool
2. Read the tool call → execute the function
3. Wrap the result in a `ToolMessage` → send it back to the LLM
4. Repeat until the LLM produces a final text answer

In [ ]:
# Step 1: Initial query
messages = [HumanMessage(content="Summarize this YouTube video in English: https://www.youtube.com/watch?v=T-D1OfcDW1M")]
response_1 = llm_with_tools.invoke(messages)

# The LLM doesn't answer directly — it asks to call a tool
tool_calls_1 = response_1.tool_calls
display(JSON(tool_calls_1))

In [ ]:
# Step 2: Execute the first tool call (extract_video_id)
messages.append(response_1)

my_tool = tool_mapping[tool_calls_1[0]['name']]
video_id = my_tool.invoke(tool_calls_1[0]['args'])
print("Extracted video ID:", video_id)

# Step 3: Feed the result back as a ToolMessage
messages.append(ToolMessage(content=video_id, tool_call_id=tool_calls_1[0]['id']))

In [ ]:
# Step 4: Second LLM call — it now decides to fetch the transcript
response_2 = llm_with_tools.invoke(messages)
messages.append(response_2)

tool_calls_2 = response_2.tool_calls
display(JSON(tool_calls_2))

In [ ]:
# Step 5: Execute the second tool call (fetch_transcript)
transcript_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
messages.append(ToolMessage(content=transcript_output, tool_call_id=tool_calls_2[0]['id']))

# Step 6: Final LLM call — generates the summary
summary = llm_with_tools.invoke(messages)
print(summary.content)

---
## 5. Automated Summarization Chain <a id='chain'></a>

The manual steps above can be composed into a reusable **LangChain chain** using the pipe operator `|`. Each step receives a dictionary (`x`), transforms it, and passes it to the next step.

`RunnablePassthrough.assign()` adds new keys to the dictionary without removing existing ones — this is how state is accumulated across steps. `RunnableLambda` at the end extracts only the final value we want.

A helper function `execute_tool` is defined first to handle the tool dispatch cleanly:

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def execute_tool(tool_call):
    """Execute a single tool call and return a ToolMessage."""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(content=str(result), tool_call_id=tool_call["id"])
    except Exception as e:
        return ToolMessage(content=f"Error: {str(e)}", tool_call_id=tool_call["id"])

In [ ]:
summarization_chain = (
    # Wrap the query string into a HumanMessage
    RunnablePassthrough.assign(
        messages=lambda x: [HumanMessage(content=x["query"])]
    )
    # First LLM call — decides to extract the video ID
    | RunnablePassthrough.assign(
        ai_response=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Execute the first tool call
    | RunnablePassthrough.assign(
        tool_messages=lambda x: [execute_tool(tc) for tc in x["ai_response"].tool_calls]
    )
    # Append the AI response and tool results to message history
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
    )
    # Second LLM call — decides to fetch the transcript
    | RunnablePassthrough.assign(
        ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Execute the second tool call
    | RunnablePassthrough.assign(
        tool_messages2=lambda x: [execute_tool(tc) for tc in x["ai_response2"].tool_calls]
    )
    # Update message history again
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
    )
    # Final LLM call — generates the summary
    | RunnablePassthrough.assign(
        summary=lambda x: llm_with_tools.invoke(x["messages"]).content
    )
    # Extract only the summary text
    | RunnableLambda(lambda x: x["summary"])
)

In [ ]:
result = summarization_chain.invoke({
    "query": "Summarize this YouTube video: https://www.youtube.com/watch?v=1bUy-1hGZpI"
})

print("Video Summary:\n", result)

---
## 6. Recursive Agent Loop <a id='recursive'></a>

The summarization chain above has a fixed structure — it always calls exactly two tools in a fixed order. A more powerful approach is a **recursive agent** that keeps calling tools until the LLM decides it has enough information to answer.

This works in a loop:
- If the LLM's last message contains tool calls → execute them and loop again
- If it contains no tool calls → it has finished, return the final answer

Three functions implement this pattern:

In [ ]:
def execute_tool(tool_call):
    """Execute a single tool call and return a ToolMessage."""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(content=str(result), tool_call_id=tool_call["id"])
    except Exception as e:
        return ToolMessage(content=f"Error: {str(e)}", tool_call_id=tool_call["id"])


def process_tool_calls(messages):
    """Execute all tool calls from the last message and get the next LLM response."""
    last_message = messages[-1]
    tool_messages = [
        execute_tool(tc)
        for tc in getattr(last_message, 'tool_calls', [])
    ]
    updated_messages = messages + tool_messages
    next_ai_response = llm_with_tools.invoke(updated_messages)
    return updated_messages + [next_ai_response]


def should_continue(messages):
    """Return True if the last message still has pending tool calls."""
    last_message = messages[-1]
    return bool(getattr(last_message, 'tool_calls', None))


def _recursive_chain(messages):
    """Recursively process tool calls until the LLM produces a final answer."""
    if should_continue(messages):
        new_messages = process_tool_calls(messages)
        return _recursive_chain(new_messages)
    return messages


recursive_chain = RunnableLambda(_recursive_chain)

The **universal chain** wires everything together. It converts the user query into a `HumanMessage`, gets the first LLM response, then hands control to the recursive loop until a final answer is produced.

In [ ]:
universal_chain = (
    RunnableLambda(lambda x: [HumanMessage(content=x["query"])])
    | RunnableLambda(lambda messages: messages + [llm_with_tools.invoke(messages)])
    | recursive_chain
)

In [ ]:
# The agent decides on its own which tools to call — no fixed sequence
response = universal_chain.invoke({
    "query": "Find the top 3 trending AI videos on YouTube and give me their metadata"
})

print(response[-1].content)

In [ ]:
# Try with a direct video URL
response = universal_chain.invoke({
    "query": "Summarize this video and tell me its stats: https://www.youtube.com/watch?v=T-D1OfcDW1M"
})

print(response[-1].content)